# 第3回：学習・検証・テストと汎化

完成コードを実行し、学習時の正解と未知データでの正解の違いを説明します。分類の代表アルゴリズムを実装する回ではなく、評価の考え方を確かめる学習用の小さな例です。標準ライブラリだけで計算し、同梱Python・JSON・外部データ・APIは読み込みません。

このNotebookだけで、カーネルを再起動して全セル実行できます。Python 3.10以降で動く記法を使っています。Notebook環境の準備は `programs/README.md` と共通の環境説明を参照してください。提出・採点条件を追加するものではありません。


## 1. データと処理コード

架空の整数の特徴と0/1ラベルを使います。IDは同じ観測を別の用途に使い回さないための印です。学習8件・検証4件・テスト4件は固定され、乱数は使いません。

しきい値モデルは、学習誤りが最少の候補を採用し、同点なら小さい値を採用します。記憶モデルは既知の特徴のラベルを記憶し、未知の特徴には学習データの多数派ラベルを返します。同数なら0です。特徴とラベルの対応や比率は説明のために作ったもので、実データの性能を表しません。


In [ ]:
"""第3回：学習・検証・テストを分けて分類を比較する（標準ライブラリ）。"""
from fractions import Fraction
import json


# 授業用の架空データ。行は (ID, 整数の特徴, 0/1のラベル)。
TRAIN = (
    ("tr01", 10, 0), ("tr02", 20, 0), ("tr03", 30, 1), ("tr04", 40, 0),
    ("tr05", 60, 1), ("tr06", 70, 1), ("tr07", 80, 1), ("tr08", 90, 1),
)
VALIDATION = (
    ("va01", 15, 0), ("va02", 35, 0), ("va03", 65, 1), ("va04", 85, 1),
)
TEST = (
    ("te01", 5, 0), ("te02", 25, 0), ("te03", 75, 1), ("te04", 95, 1),
)


def validate_rows(rows):
    """この教材の入力条件を確認する。ラベルの正しさ自体は保証しない。"""
    if not rows:
        raise ValueError("データは1件以上必要です")
    ids = set()
    for row in rows:
        if len(row) != 3:
            raise ValueError("各行は (ID, 特徴, ラベル) にしてください")
        identifier, feature, label = row
        if not isinstance(identifier, str) or not identifier or identifier in ids:
            raise ValueError("IDは空でない文字列で、重複しないようにしてください")
        if type(feature) is not int or type(label) is not int or label not in (0, 1):
            raise ValueError("特徴は整数、ラベルは整数の0または1にしてください")
        ids.add(identifier)


def validate_splits(train, validation, test):
    """同じIDの使い回しを検出する。別IDの類似データ漏洩は別途点検が必要。"""
    seen = set()
    for rows in (train, validation, test):
        validate_rows(rows)
        ids = {row[0] for row in rows}
        if seen & ids:
            raise ValueError("学習・検証・テストで同じIDを使わないでください")
        seen.update(ids)


def fit_threshold(train):
    """学習誤り最小のしきい値を学習。同点は小さいしきい値を採用。"""
    validate_rows(train)
    values = sorted({x for _, x, _ in train})
    candidates = [Fraction(values[0] - 1)]
    candidates += [Fraction(a + b, 2) for a, b in zip(values, values[1:])]
    candidates += [Fraction(values[-1] + 1)]
    threshold = min(candidates, key=lambda t: (
        sum(int(x >= t) != label for _, x, label in train), t,
    ))
    return {"threshold": threshold}


def fit_memory(train):
    """特徴とラベルを記憶。未知の特徴は学習多数派、同数なら0。"""
    validate_rows(train)
    labels = {}
    for _, x, label in train:
        if x in labels and labels[x] != label:
            raise ValueError("記憶モデルでは同じ特徴に相反するラベルを記憶できません")
        labels[x] = label
    ones = sum(label for _, _, label in train)
    return {"labels": labels, "fallback": int(ones > len(train) - ones)}


def predict(model, feature):
    """入力の特徴だけから予測。正解ラベルは受け取らない。"""
    if type(feature) is not int:
        raise ValueError("予測する特徴は整数にしてください")
    if "threshold" in model:
        return int(feature >= model["threshold"])
    return model["labels"].get(feature, model["fallback"])


def evaluate(model, rows):
    """予測してからラベルと比較。正解数と総数をそのまま返す。"""
    validate_rows(rows)
    predictions = [predict(model, x) for _, x, _ in rows]
    correct = sum(prediction == row[2] for prediction, row in zip(predictions, rows))
    return {"predictions": predictions, "correct": correct, "total": len(rows)}


def choose_model(validation_scores):
    """検証結果で選択。同点ならthreshold。テスト結果は受け取らない。"""
    def accuracy(name):
        score = validation_scores[name]
        return Fraction(score["correct"], score["total"])
    return "threshold" if accuracy("threshold") >= accuracy("memory") else "memory"


def run_experiment(train=TRAIN, validation=VALIDATION, test=TEST):
    """学習→検証で選択→固定してテスト。原データは変更しない。"""
    validate_splits(train, validation, test)
    models = {"threshold": fit_threshold(train), "memory": fit_memory(train)}
    scores = {name: {
        "training": evaluate(model, train),
        "validation": evaluate(model, validation),
    } for name, model in models.items()}
    selected = choose_model({name: score["validation"] for name, score in scores.items()})
    # 選択を確定した後、比較教材として両モデルのテスト結果を表示する。
    # この表示を見てモデルや設定を選び直さない。
    for name, model in models.items():
        scores[name]["test"] = evaluate(model, test)
    scores["threshold"]["threshold"] = str(models["threshold"]["threshold"])
    scores["memory"]["fallback"] = models["memory"]["fallback"]
    return {
        "data": {"training": [list(row) for row in train],
                 "validation": [list(row) for row in validation],
                 "test": [list(row) for row in test]},
        "models": scores,
        "selected_model": selected,
        "selected_test": scores[selected]["test"],
    }


def print_result(result):
    names = {"threshold": "しきい値モデル", "memory": "記憶モデル"}
    print("架空の固定データ：学習8件・検証4件・テスト4件")
    print("学習したしきい値:", result["models"]["threshold"]["threshold"])
    for name, scores in result["models"].items():
        parts = [f"{title} {scores[key]['correct']}/{scores[key]['total']}"
                 for key, title in (("training", "学習"), ("validation", "検証"), ("test", "テスト"))]
        print(names[name] + ": " + " / ".join(parts))
    print("検証で選択:", names[result["selected_model"]])
    print("テスト結果を見て選び直さず、未知のデータでの限界を読みます。")


## 2. 基準結果を確かめる

学習データだけでモデルを作り、検証の正解率で選びます。同点ならしきい値モデルを採用します。選択を確定した後でテストします。比較教材として両モデルのテスト結果も表示しますが、それを見て選び直してはいけません。

予想してから実行しましょう。学習時に全問正解したモデルが、検証でも優れるでしょうか？


In [ ]:
baseline_result = run_experiment()
print_result(baseline_result)


## 3. 予測列と誤りを読む

想定結果：しきい値は25。しきい値モデルは学習7/8、検証3/4、テスト3/4。記憶モデルは学習8/8、検証2/4、テスト2/4。検証でしきい値モデルを選び、その最終テストは3/4です。

しきい値モデルは検証の特徴35とテストの特徴25を誤ります。25以上を1にするためです。記憶モデルは未知の特徴をすべて多数派の1にし、検証・テストともラベル0の2件を誤ります。これは特定の架空例です。「この手法なら常に75%」とは言えません。


In [ ]:
for name, scores in baseline_result["models"].items():
    print(name)
    for split in ("training", "validation", "test"):
        score = scores[split]
        print(split, score["predictions"],
              "正解率", Fraction(score["correct"], score["total"]))


## 4. 入力だけを変える

学習済みモデルを固定したまま24、25、26を予測します。しきい値モデルは `[0, 1, 1]`、記憶モデルは未知の値なので `[1, 1, 1]` になります。これは推論です。入力を変えて予測するだけでは再学習しません。


In [ ]:
threshold_model = fit_threshold(TRAIN)
memory_model = fit_memory(TRAIN)
new_inputs = [24, 25, 26]  # このリストの値を変えて比較
print("しきい値:", [predict(threshold_model, x) for x in new_inputs])
print("記憶:", [predict(memory_model, x) for x in new_inputs])


## 5. 学習ラベルを一つ変える

元データは保持し、tr03のラベルを1から0へ変えた別の学習例を作ります。新たなしきい値は50、学習正解数は8/8、検証正解数は4/4になります。ラベルの変更が学習結果を変えることを観察します。実データでこの変更が正しいという意味ではありません。

変更例は学習・検証だけで観察します。すでに見た最終テストを繰り返し見ながら改善する手順にはしません。


In [ ]:
changed_train = [(identifier, x, 0 if identifier == "tr03" else label)
                 for identifier, x, label in TRAIN]
changed_model = fit_threshold(changed_train)
print("変更後のしきい値:", changed_model["threshold"])
print("学習:", evaluate(changed_model, changed_train))
print("検証:", evaluate(changed_model, VALIDATION))
print("元のtr03:", TRAIN[2])


## 6. 学習データの構成を変える

tr03を除いた学習集合をコピーして比較します。しきい値は50になります。記憶モデルはラベル0が3件、1が4件なので未知入力に1を返します。

行の追加・削除を試すときはIDを重複させず、検証・テストの観測を学習側へ移して評価を良く見せないでください。この検査は同一IDの重複を検出するだけで、別IDにした同じ対象や似た記録を自動的に見抜くものではありません。


In [ ]:
smaller_train = [row for row in TRAIN if row[0] != "tr03"]
validate_splits(smaller_train, VALIDATION, TEST)
for name, model in (("しきい値", fit_threshold(smaller_train)),
                    ("記憶", fit_memory(smaller_train))):
    print(name, model)
    print("検証:", evaluate(model, VALIDATION))


## まとめ

1. どの値がデータから決まり、どの値が人の決めた規則でしたか？
2. 記憶モデルの学習正解率100%だけで、未知の入力への対応を判断できますか？
3. 検証用と最終テスト用の役割はどう違いますか？

正解数と総数で比較し、有理数で正解率を計算しました。浮動小数点の誤差や乱数はありません。Python版とこのNotebookの基準データ・予測列・正解数・選択結果は完全一致で検証します。学生の実行環境や授業での理解度は別に確認が必要です。
